# Validation: LLM Responses without XAI (N_SAMPLES=40)

This notebook validates LLM analysis outputs for the without-XAI experiment with 40 training/prediction samples.

In [1]:
import json
import re
from collections import defaultdict

with open('resultados_without_xai_samples_40.json', 'r', encoding='utf-8') as f:
    results = json.load(f)

print(f"Loaded {len(results)} model responses")
print(f"Models: {list(results.keys())}")

Loaded 4 model responses
Models: ['glm-4.7-flash:latest', 'qwen3:14b', 'gpt-oss:20b', 'qwen3:30b']


In [2]:
GROUND_TRUTH = {
    'BotAttack': ['Port', 'Status', 'Payload_Size'],
    'Normal': ['Port', 'Status', 'Payload_Size'],
    'PortScan': ['Payload_Size', 'Status', 'Port'],
}

print("Ground Truth SHAP Rankings:")
for cls, features in GROUND_TRUTH.items():
    print(f"  {cls}: {features}")

Ground Truth SHAP Rankings:
  BotAttack: ['Port', 'Status', 'Payload_Size']
  Normal: ['Port', 'Status', 'Payload_Size']
  PortScan: ['Payload_Size', 'Status', 'Port']


In [3]:
def extract_feature_rankings(text):
    features = ['Port', 'Status', 'Payload_Size', 'Request_Type', 'Protocol', 'User_Agent']
    feature_mentions = defaultdict(int)
    
    for feat in features:
        importance_contexts = ['important', 'significant', 'key', 'main', 'primary', 'drives', 'influences', 'impact']
        for ctx in importance_contexts:
            pattern = feat + r'[^.]*' + ctx + r'[^.]|' + ctx + r'[^.]' + feat
            if re.search(pattern, text, re.IGNORECASE):
                feature_mentions[feat] += 1
    
    rankings = {}
    text_lower = text.lower()
    for feat in features:
        if any(phrase in text_lower for phrase in [f'{feat.lower()} most', f'most {feat.lower()}', f'primary {feat.lower()}']):
            rankings[feat] = 'top'
    
    return dict(feature_mentions), rankings


def check_shap_citation(text):
    return {
        'has_shap': bool(re.search(r'shap', text, re.IGNORECASE)),
        'has_lime': bool(re.search(r'lime', text, re.IGNORECASE)),
        'inappropriate_xai_citation': bool(re.search(r'shap|lime', text, re.IGNORECASE))
    }


def detect_fabrication(text):
    fabrications = []
    metric_claims = re.findall(r'(?:precision|recall|f1|f-score)\s*[=:]\s*[\d.]+', text, re.IGNORECASE)
    if metric_claims:
        fabrications.append(('Per-class metrics claimed', metric_claims))
    return fabrications


def validate_response(model_name, text):
    mentions, rankings = extract_feature_rankings(text)
    xai_check = check_shap_citation(text)
    fabrications = detect_fabrication(text)
    
    user_agent_overvalued = mentions.get("User_Agent", 0) > 0 or "User_Agent" in rankings
    top3_mentioned = sorted(mentions.items(), key=lambda x: -x[1])[:3]
    top3_features = [f[0] for f in top3_mentioned]
    
    has_port_top = mentions.get("Port", 0) == max(mentions.values()) if mentions else False
    has_correct_top3 = 'Port' in top3_features and 'Status' in top3_features
    
    return {
        'model': model_name,
        'char_count': len(text),
        'feature_mentions': mentions,
        'top3_features': top3_features,
        'user_agent_overvalued': user_agent_overvalued,
        'xai_citation': xai_check,
        'fabrications': fabrications,
        'has_port_top': has_port_top,
        'has_correct_top3': has_correct_top3
    }

In [4]:
validations = {}
for model, text in results.items():
    print(f"\n{'='*60}")
    print(f"Validating: {model}")
    print(f"{'='*60}")
    validations[model] = validate_response(model, text)
    
    print(f"Character count: {validations[model]['char_count']}")
    print(f"Top-3 features: {validations[model]['top3_features']}")
    print(f"User_Agent overvalued: {validations[model]['user_agent_overvalued']}")
    print(f"Correct top-3: {validations[model]['has_correct_top3']}")


Validating: glm-4.7-flash:latest
Character count: 7141
Top-3 features: ['Payload_Size']
User_Agent overvalued: False
Correct top-3: False

Validating: qwen3:14b
Character count: 6068
Top-3 features: ['User_Agent']
User_Agent overvalued: True
Correct top-3: False

Validating: gpt-oss:20b
Character count: 10452
Top-3 features: []
User_Agent overvalued: False
Correct top-3: False

Validating: qwen3:30b
Character count: 8249
Top-3 features: ['Port', 'Status', 'User_Agent']
User_Agent overvalued: True
Correct top-3: True


In [5]:
import pandas as pd

summary_data = []
for model, val in validations.items():
    summary_data.append({
        'Model': model,
        'Response Length': val['char_count'],
        'Port as Top Feature': val['has_port_top'],
        'Correct Top-3 (Port+Status)': val['has_correct_top3'],
        'User_Agent Overvalued': val['user_agent_overvalued'],
        'Inappropriate XAI Citation': val['xai_citation']['inappropriate_xai_citation'],
        'Fabrications Detected': len(val['fabrications']) > 0
    })

df_summary = pd.DataFrame(summary_data)
print("\nValidation Summary (N_SAMPLES=40):")
print(df_summary.to_string(index=False))


Validation Summary (N_SAMPLES=40):
               Model  Response Length  Port as Top Feature  Correct Top-3 (Port+Status)  User_Agent Overvalued  Inappropriate XAI Citation  Fabrications Detected
glm-4.7-flash:latest             7141                False                        False                  False                        True                  False
           qwen3:14b             6068                False                        False                   True                        True                  False
         gpt-oss:20b            10452                False                        False                  False                       False                  False
           qwen3:30b             8249                 True                         True                   True                        True                  False


In [6]:
print("\n" + "="*60)
print("KEY FINDINGS")
print("="*60)

correct_count = sum(1 for v in validations.values() if v['has_correct_top3'])
ua_bias_count = sum(1 for v in validations.values() if v['user_agent_overvalued'])

print(f"\n1. Feature Ranking Accuracy: {correct_count}/4 correct")
print(f"2. User_Agent Bias: {ua_bias_count}/4 models")

xai_count = sum(1 for v in validations.values() if v['xai_citation']['inappropriate_xai_citation'])
print(f"3. Inappropriate XAI Citations: {xai_count}/4")

fab_count = sum(1 for v in validations.values() if len(v['fabrications']) > 0)
print(f"4. Fabrications: {fab_count}/4")


KEY FINDINGS

1. Feature Ranking Accuracy: 1/4 correct
2. User_Agent Bias: 2/4 models
3. Inappropriate XAI Citations: 3/4
4. Fabrications: 0/4
